# 01 - Data Understanding & Ingestion
## NASA C-MAPSS Turbofan Engine Degradation Simulation Dataset (FD001)

### Project Context
- **Project Title**: Machine Learning-Based Remaining Useful Life Estimation for Turbofan Engines
- **Task Type**: Supervised Regression
- **Target Variable**: Remaining Useful Life (RUL) in operational flight cycles
- **Dataset Subset**: FD001 (Sea level conditions, High-Pressure Compressor failure mode)

### Notebook Objectives
1. Ingest raw space-separated text files via modular data loader (`src/data_loader.py`).
2. Verify dimensional shapes and unique engine counts across training, test, and ground truth sets.
3. Inspect schema types, first/last rows, and memory footprints.
4. Compute descriptive statistics across all 26 telemetry channels.
5. Audit data integrity: check for missing values, duplicate rows, and duplicate cycle indices.
6. Verify no data leakage or premature transformations occur at this stage.

*Note: In accordance with project requirements, no ML models are trained, and raw values remain unmodified.*

In [1]:
import sys
from pathlib import Path

# Ensure project root is in Python path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
from src.data_loader import load_cmapss_data

# Set display options
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)

print(f"Project root identified at: {project_root.resolve()}")

Project root identified at: F:\Turbofan_RUL_Project


---
## 1. Load Raw Dataset via `src/data_loader.py`

We invoke the centralized data loader function `load_cmapss_data()`. This function parses the space-delimited text files, trims any accidental spacing-induced columns, assigns standardized column headers, and leaves all numerical values intact.

In [2]:
train_df, test_df, rul_df = load_cmapss_data(data_dir=project_root / "data" / "raw")
print("Dataset successfully ingested via src.data_loader!")

Dataset successfully ingested via src.data_loader!


---
## 2. Dataset Dimensions & Unique Engine Counts

In [3]:
print("=" * 60)
print("                   DATASET DIMENSIONS")
print("=" * 60)
print(f"Training Trajectories (train_FD001.txt) : {train_df.shape[0]:,} rows x {train_df.shape[1]} columns")
print(f"Test Trajectories (test_FD001.txt)       : {test_df.shape[0]:,} rows x {test_df.shape[1]} columns")
print(f"Ground Truth RUL (RUL_FD001.txt)        : {rul_df.shape[0]:,} rows x {rul_df.shape[1]} column")

print("\n" + "=" * 60)
print("                 UNIQUE ENGINE UNITS")
print("=" * 60)
print(f"Unique engines in Training Set : {train_df['unit_number'].nunique()} (Engine IDs: {train_df['unit_number'].min()} to {train_df['unit_number'].max()})")
print(f"Unique engines in Test Set     : {test_df['unit_number'].nunique()} (Engine IDs: {test_df['unit_number'].min()} to {test_df['unit_number'].max()})")
print(f"Target evaluations in RUL Set  : {len(rul_df)} ground truth values")

                   DATASET DIMENSIONS
Training Trajectories (train_FD001.txt) : 20,631 rows x 26 columns
Test Trajectories (test_FD001.txt)       : 13,096 rows x 26 columns
Ground Truth RUL (RUL_FD001.txt)        : 100 rows x 1 column

                 UNIQUE ENGINE UNITS
Unique engines in Training Set : 100 (Engine IDs: 1 to 100)
Unique engines in Test Set     : 100 (Engine IDs: 1 to 100)
Target evaluations in RUL Set  : 100 ground truth values


---
## 3. Data Preview: First and Last 5 Rows

Examining snapshots of the beginning and end of the dataset allows us to verify row structure and telemetry formatting.

In [4]:
print("--- FIRST 5 ROWS (Training Set) ---")
train_df.head()

--- FIRST 5 ROWS (Training Set) ---


,unit_number,time_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [5]:
print("--- LAST 5 ROWS (Training Set) ---")
train_df.tail()

--- LAST 5 ROWS (Training Set) ---


,unit_number,time_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
20626,100,196,-0.0004,-0.0003,100.0,518.67,643.49,1597.98,1428.63,14.62,21.61,551.43,2388.19,9065.52,1.3,48.07,519.49,2388.26,8137.60,8.4956,0.03,397,2388,100.0,38.49,22.9735
20627,100,197,-0.0016,-0.0005,100.0,518.67,643.54,1604.50,1433.58,14.62,21.61,550.86,2388.23,9065.11,1.3,48.04,519.68,2388.22,8136.50,8.5139,0.03,395,2388,100.0,38.30,23.1594
20628,100,198,0.0004,0.0000,100.0,518.67,643.42,1602.46,1428.18,14.62,21.61,550.94,2388.24,9065.90,1.3,48.09,520.01,2388.24,8141.05,8.5646,0.03,398,2388,100.0,38.44,22.9333
20629,100,199,-0.0011,0.0003,100.0,518.67,643.23,1605.26,1426.53,14.62,21.61,550.68,2388.25,9073.72,1.3,48.39,519.67,2388.23,8139.29,8.5389,0.03,395,2388,100.0,38.29,23.0640
20630,100,200,-0.0032,-0.0005,100.0,518.67,643.85,1600.38,1432.14,14.62,21.61,550.79,2388.26,9061.48,1.3,48.20,519.30,2388.26,8137.33,8.5036,0.03,396,2388,100.0,38.37,23.0522


---
## 4. Column Data Types & Memory Usage

In [6]:
print("--- DATA TYPES & NON-NULL SUMMARY ---")
train_df.info()

memory_mb = train_df.memory_usage().sum() / (1024 ** 2)
print(f"\nTotal DataFrame Memory Footprint: {memory_mb:.2f} MB")

--- DATA TYPES & NON-NULL SUMMARY ---
<class 'pandas.DataFrame'>
RangeIndex: 20631 entries, 0 to 20630
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   unit_number            20631 non-null  int64  
 1   time_cycles            20631 non-null  int64  
 2   operational_setting_1  20631 non-null  float64
 3   operational_setting_2  20631 non-null  float64
 4   operational_setting_3  20631 non-null  float64
 5   sensor_1               20631 non-null  float64
 6   sensor_2               20631 non-null  float64
 7   sensor_3               20631 non-null  float64
 8   sensor_4               20631 non-null  float64
 9   sensor_5               20631 non-null  float64
 10  sensor_6               20631 non-null  float64
 11  sensor_7               20631 non-null  float64
 12  sensor_8               20631 non-null  float64
 13  sensor_9               20631 non-null  float64
 14  sensor_10              2063

---
## 5. Descriptive Statistics

We inspect the five-number summary (mean, std, min, 25%, 50%, 75%, max) across operational settings and 21 sensors.

In [7]:
train_df.describe().T

,count,mean,std,min,25%,50%,75%,max
unit_number,20631.0,51.506568,2.922763e+01,1.0000,26.0000,52.0000,77.0000,100.0000
time_cycles,20631.0,108.807862,6.888099e+01,1.0000,52.0000,104.0000,156.0000,362.0000
operational_setting_1,20631.0,-0.000009,2.187313e-03,-0.0087,-0.0015,0.0000,0.0015,0.0087
operational_setting_2,20631.0,0.000002,2.930621e-04,-0.0006,-0.0002,0.0000,0.0003,0.0006
operational_setting_3,20631.0,100.000000,0.000000e+00,100.0000,100.0000,100.0000,100.0000,100.0000
sensor_1,20631.0,518.670000,0.000000e+00,518.6700,518.6700,518.6700,518.6700,518.6700
sensor_2,20631.0,642.680934,5.000533e-01,641.2100,642.3250,642.6400,643.0000,644.5300
sensor_3,20631.0,1590.523119,6.131150e+00,1571.0400,1586.2600,1590.1000,1594.3800,1616.9100
sensor_4,20631.0,1408.933782,9.000605e+00,1382.2500,1402.3600,1408.0400,1414.5550,1441.4900
sensor_5,20631.0,14.620000,5.329200e-15,14.6200,14.6200,14.6200,14.6200,14.6200


---
## 6. Operating Cycles per Engine (Lifetime Analysis)

In the training set, each engine trajectory represents a continuous run-to-failure cycle. The final recorded cycle for each engine is its operational lifespan.

In [8]:
cycles_per_unit = train_df.groupby('unit_number')['time_cycles'].max()

print("=" * 60)
print("             ENGINE LIFETIME SUMMARY (TRAINING SET)")
print("=" * 60)
print(f"Shortest Engine Lifetime : {cycles_per_unit.min()} cycles (Engine #{cycles_per_unit.idxmin()})")
print(f"Longest Engine Lifetime  : {cycles_per_unit.max()} cycles (Engine #{cycles_per_unit.idxmax()})")
print(f"Mean Engine Lifetime     : {cycles_per_unit.mean():.2f} cycles")
print(f"Median Engine Lifetime   : {cycles_per_unit.median():.1f} cycles")
print(f"Standard Deviation       : {cycles_per_unit.std():.2f} cycles")
print("-" * 60)
print("Sample of First 10 Engine Lifespans:")
for unit, max_c in cycles_per_unit.head(10).items():
    print(f"  Engine {unit:3d}: {max_c} operational cycles")

             ENGINE LIFETIME SUMMARY (TRAINING SET)
Shortest Engine Lifetime : 128 cycles (Engine #39)
Longest Engine Lifetime  : 362 cycles (Engine #69)
Mean Engine Lifetime     : 206.31 cycles
Median Engine Lifetime   : 199.0 cycles
Standard Deviation       : 46.34 cycles
------------------------------------------------------------
Sample of First 10 Engine Lifespans:
  Engine   1: 192 operational cycles
  Engine   2: 287 operational cycles
  Engine   3: 179 operational cycles
  Engine   4: 189 operational cycles
  Engine   5: 269 operational cycles
  Engine   6: 188 operational cycles
  Engine   7: 259 operational cycles
  Engine   8: 150 operational cycles
  Engine   9: 201 operational cycles
  Engine  10: 222 operational cycles


---
## 7. Data Quality & Integrity Audits

Checking for missing values (`NaN`), duplicate rows, and index collisions between `unit_number` and `time_cycles`.

In [9]:
print("=" * 60)
print("                 DATA INTEGRITY CHECKS")
print("=" * 60)

# Check for nulls
null_train = train_df.isna().sum().sum()
null_test = test_df.isna().sum().sum()
null_rul = rul_df.isna().sum().sum()
print(f"Missing values in Training set : {null_train}")
print(f"Missing values in Test set     : {null_test}")
print(f"Missing values in RUL set      : {null_rul}")

# Check for duplicate rows
dup_train = train_df.duplicated().sum()
dup_test = test_df.duplicated().sum()
dup_rul = rul_df.duplicated().sum()
print(f"Duplicate records in Training  : {dup_train}")
print(f"Duplicate records in Test      : {dup_test}")
print(f"Duplicate records in RUL       : {dup_rul}")

# Check for duplicate (unit_number, time_cycles) pairs
pair_dups = train_df.duplicated(subset=['unit_number', 'time_cycles']).sum()
print(f"Duplicate (unit, cycle) pairs  : {pair_dups}")

assert null_train == 0, "Error: Unexpected missing values detected in training set!"
assert dup_train == 0, "Error: Duplicate rows found in training set!"
assert pair_dups == 0, "Error: Non-unique engine cycle snapshots detected!"
print("\nAll integrity assertions passed successfully!")

                 DATA INTEGRITY CHECKS
Missing values in Training set : 0
Missing values in Test set     : 0
Missing values in RUL set      : 0
Duplicate records in Training  : 0
Duplicate records in Test      : 0
Duplicate records in RUL       : 29
Duplicate (unit, cycle) pairs  : 0

All integrity assertions passed successfully!


---
## 8. Summary of Findings (Data Understanding)

| Property | Value / Observation | ML Implication |
| :--- | :--- | :--- |
| **Training Records** | 20,631 cycles across 100 engines | Sufficient volume for regression modeling |
| **Test Records** | 13,096 cycles across 100 engines | Test data must remain unseen during feature engineering |
| **Engine Lifespans** | Min: 128 cycles, Max: 362 cycles, Mean: 206.31 | Variable engine wear rates, requiring normalized or sequence-aware features |
| **Missing Values** | Exactly 0 across all files | No imputation required |
| **Duplicates** | Exactly 0 duplicate records | Clean telemetry snapshots |
| **Data Types** | Integer (IDs, cycles), Float (Settings, Sensors) | Numeric representations ready for numerical processing |

*Next Step: Proceed to `02_eda.ipynb` to perform comprehensive exploratory data analysis, evaluate sensor variances, and observe degradation signatures.*